# COMP6008 Group 17 – Agent 1: Tabular Q-learning

This notebook trains a tabular Q-learning agent on the home energy scheduling environment (`EnergyEnv`). The implementation is adapted from the `Qlearning` function in **Practical 5 (Temporal-difference Learning for Control)**, with three changes needed for this environment:

1. **Discretiser:** the environment emits a continuous 15-value observation, so it is mapped to a discrete state index before using the Q-table.
2. **ε-greedy action selection per step:** Practical 5 rebuilds the full behaviour-policy table after every step. That costs O(|S|·|A|) per step, which is fine for Cliff Walking (48 states) but too slow for 4,320 states. Here ε-greedy is applied only to the current state, which gives the identical behaviour policy at O(|A|) cost.
3. **Decaying ε:** ε decreases linearly from 1.0 to 0.05, following the GLIE idea from Practical 5, Exercise 1, so the agent explores early and exploits later.

**How to run:** Runtime → Run all. The main training run takes about 3 minutes in Colab; the hyperparameter sweep in Section 8 takes about 15–20 minutes.

## 0. Setup
Install Gymnasium and write the environment files (identical to the environment notebook), so this notebook runs on its own.

In [ ]:
!pip install -q gymnasium

In [ ]:
%%writefile profiles.py
"""
profiles.py - Synthetic hourly data for the home energy scheduling environment.

Everything here is a 24-value array (one value per hour, index 0 = midnight).
All numbers are illustrative assumptions for a typical Perth household;
state them as assumptions in the report.

Usage:
    from profiles import sample_day
    day = sample_day(rng)          # rng = np.random.default_rng(seed)
    day["solar"], day["load"], day["price"], ...
"""

import numpy as np

HOURS = np.arange(24)

# ---------------------------------------------------------------------------
# Solar generation (kW) - bell curve between sunrise and sunset, scaled by weather
# ---------------------------------------------------------------------------
SOLAR_PEAK_KW = 5.0            # roughly a 6.6 kW rooftop system on a clear day
SUNRISE, SUNSET = 6, 18

WEATHER_TYPES = ["sunny", "cloudy", "rainy"]
WEATHER_PROBS = [0.6, 0.3, 0.1]                          # Perth is mostly sunny
WEATHER_SCALE = {"sunny": 1.0, "cloudy": 0.5, "rainy": 0.2}


def solar_profile(weather, rng=None, noise=0.1):
    """Hourly solar output (kW) for a given weather type."""
    curve = np.zeros(24)
    daylight = (HOURS >= SUNRISE) & (HOURS <= SUNSET)
    # half-sine shape: 0 at sunrise/sunset, peak at midday
    curve[daylight] = np.sin(np.pi * (HOURS[daylight] - SUNRISE) / (SUNSET - SUNRISE))
    solar = SOLAR_PEAK_KW * WEATHER_SCALE[weather] * curve
    if rng is not None:                                   # small random variation
        solar *= 1 + rng.normal(0, noise, 24)
    return np.clip(solar, 0, None)


# ---------------------------------------------------------------------------
# Household base load (kW) - things we do NOT control (fridge, lights, cooking)
# ---------------------------------------------------------------------------
BASE_LOAD = np.array([
    0.4, 0.3, 0.3, 0.3, 0.3, 0.4,    # 00-05 night
    0.8, 1.2, 1.0, 0.6, 0.5, 0.5,    # 06-11 morning peak
    0.6, 0.5, 0.5, 0.6, 0.9, 1.5,    # 12-17 afternoon
    2.0, 1.8, 1.4, 1.0, 0.7, 0.5,    # 18-23 evening peak
])


def load_profile(rng=None, noise=0.1):
    load = BASE_LOAD.copy()
    if rng is not None:
        load *= 1 + rng.normal(0, noise, 24)
    return np.clip(load, 0.1, None)


# ---------------------------------------------------------------------------
# Time-of-use tariff ($/kWh) - cheap midday, expensive evening peak
# ---------------------------------------------------------------------------
PRICE_OFFPEAK, PRICE_MIDDAY, PRICE_PEAK = 0.25, 0.10, 0.55
FEED_IN_PRICE = 0.03            # paid for exporting solar to the grid


def price_profile():
    price = np.full(24, PRICE_OFFPEAK)
    price[9:15] = PRICE_MIDDAY       # 9am-3pm
    price[15:21] = PRICE_PEAK        # 3pm-9pm
    return price


# ---------------------------------------------------------------------------
# Home battery and EV
# ---------------------------------------------------------------------------
BATTERY = {"capacity_kwh": 10.0, "max_rate_kw": 5.0, "efficiency": 0.95}

EV = {
    "capacity_kwh": 60.0,
    "max_rate_kw": 7.0,
    "arrive_hour": 18,          # plugged in from 6pm ...
    "depart_hour": 7,           # ... until 7am next morning
    "arrive_soc": 0.4,          # 40% charge when it gets home
    "target_soc": 0.8,          # needs 80% by departure (penalty if not)
}


def ev_plugged_in():
    """Boolean array: True for hours the EV is at home and can charge."""
    return (HOURS >= EV["arrive_hour"]) | (HOURS < EV["depart_hour"])


# ---------------------------------------------------------------------------
# Shiftable appliances - must run once per day, agent chooses when
# ---------------------------------------------------------------------------
APPLIANCES = {
    "washing_machine": {"power_kw": 1.0, "duration_h": 2, "earliest": 7,  "deadline": 22},
    "dishwasher":      {"power_kw": 1.2, "duration_h": 2, "earliest": 8,  "deadline": 23},
}


# ---------------------------------------------------------------------------
# One episode = one simulated day
# ---------------------------------------------------------------------------
def sample_day(rng, weather=None):
    """Return all profiles for one day. Weather is random unless given."""
    if weather is None:
        weather = rng.choice(WEATHER_TYPES, p=WEATHER_PROBS)
    return {
        "weather": weather,
        "solar": solar_profile(weather, rng),
        "load": load_profile(rng),
        "price": price_profile(),
        "feed_in": FEED_IN_PRICE,
        "ev_plugged_in": ev_plugged_in(),
    }

In [ ]:
%%writefile energy_env.py
"""
energy_env.py - Home energy scheduling environment (Gymnasium).

One episode = one day, 24 hourly steps, from 07:00 to 07:00 the next morning.
Starting at 07:00 means the EV's deadline (charged by 07:00 departure) falls
exactly at the end of the episode.

Observation (15 values, fixed order - do not change without telling the group):
    0 hour_sin, 1 hour_cos, 2 battery_soc, 3 ev_soc, 4 ev_plugged_in,
    5 solar_now, 6 solar_next, 7 load_now, 8 price_now, 9 price_next,
    10 sunny, 11 cloudy, 12 rainy, 13 washer_pending, 14 dishwasher_pending

Action (Discrete 12):  action = battery * 4 + ev * 2 + appliance
    battery:   0 = charge, 1 = idle, 2 = discharge
    ev:        0 = wait,   1 = charge
    appliance: 0 = defer,  1 = start next pending appliance
"""

import numpy as np
import gymnasium as gym
from gymnasium import spaces

import profiles as P

START_HOUR = 7
EV_PENALTY_PER_KWH = 1.0      # $ per kWh the EV is short at departure
APPLIANCE_PENALTY = 2.0       # $ if an appliance misses its deadline


def decode_action(action):
    """Split the single action number into its three parts."""
    action = int(action)
    return action // 4, (action // 2) % 2, action % 2


class EnergyEnv(gym.Env):
    metadata = {"render_modes": []}

    def __init__(self, weather=None):
        super().__init__()
        self.fixed_weather = weather          # None = random weather each episode
        self.appliance_names = list(P.APPLIANCES)

        low = np.array([-1, -1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=np.float32)
        high = np.array([1, 1, 1, 1, 1,
                         P.SOLAR_PEAK_KW * 1.5, P.SOLAR_PEAK_KW * 1.5, 5.0,
                         P.PRICE_PEAK, P.PRICE_PEAK,
                         1, 1, 1, 1, 1], dtype=np.float32)
        self.observation_space = spaces.Box(low, high, dtype=np.float32)
        self.action_space = spaces.Discrete(12)

    # ------------------------------------------------------------------ reset
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)                    # sets self.np_random
        self.day = P.sample_day(self.np_random, self.fixed_weather)
        self.t = 0                                   # step counter 0..23
        self.battery_soc = 0.5
        self.ev_soc = P.EV["arrive_soc"]
        # per appliance: "pending" (not started), "running", "done", "missed"
        self.app_status = {n: "pending" for n in self.appliance_names}
        self.app_hours_left = {n: 0 for n in self.appliance_names}
        return self._obs(), {}

    # ------------------------------------------------------------------- step
    def step(self, action):
        assert self.action_space.contains(action)
        h = self.hour
        b_act, ev_act, app_act = decode_action(int(action))
        solar = self.day["solar"][h]
        load = self.day["load"][h]
        price = self.day["price"][h]
        penalty = 0.0

        # 1. Appliances: start the next pending one if allowed, then run them
        if app_act == 1:
            for n in self.appliance_names:
                spec = P.APPLIANCES[n]
                if self.app_status[n] == "pending" and h >= spec["earliest"]:
                    self.app_status[n] = "running"
                    self.app_hours_left[n] = spec["duration_h"]
                    break
        appliance_kwh = 0.0
        for n in self.appliance_names:
            if self.app_status[n] == "running":
                appliance_kwh += P.APPLIANCES[n]["power_kw"]
                self.app_hours_left[n] -= 1
                if self.app_hours_left[n] == 0:
                    self.app_status[n] = "done"

        # 2. EV arrives home at 18:00 with a fixed charge level
        if h == P.EV["arrive_hour"]:
            self.ev_soc = P.EV["arrive_soc"]
        ev_kwh = 0.0
        if ev_act == 1 and self.day["ev_plugged_in"][h]:
            ev_kwh = min(P.EV["max_rate_kw"], (1 - self.ev_soc) * P.EV["capacity_kwh"])
            self.ev_soc += ev_kwh / P.EV["capacity_kwh"]

        # 3. Home battery (efficiency loss on the way in and out)
        cap, rate, eff = (P.BATTERY[k] for k in ("capacity_kwh", "max_rate_kw", "efficiency"))
        batt_in = batt_out = 0.0
        if b_act == 0:                                   # charge
            batt_in = min(rate, (1 - self.battery_soc) * cap / eff)
            self.battery_soc += batt_in * eff / cap
        elif b_act == 2:                                 # discharge
            drawn = min(rate, self.battery_soc * cap)
            self.battery_soc -= drawn / cap
            batt_out = drawn * eff
        self.battery_soc = float(np.clip(self.battery_soc, 0, 1))

        # 4. Energy balance: positive = buy from grid, negative = sell
        net = load + appliance_kwh + ev_kwh + batt_in - solar - batt_out
        grid_import, grid_export = max(net, 0.0), max(-net, 0.0)
        cost = grid_import * price - grid_export * self.day["feed_in"]

        # 5. Penalties: appliance past its latest start time
        for n in self.appliance_names:
            spec = P.APPLIANCES[n]
            latest_start = spec["deadline"] - spec["duration_h"]
            if self.app_status[n] == "pending" and h >= latest_start:
                self.app_status[n] = "missed"
                penalty += APPLIANCE_PENALTY

        # 6. Advance time; at 07:00 the EV leaves - penalise any shortfall
        self.t += 1
        terminated = self.t == 24
        if terminated:
            shortfall = max(P.EV["target_soc"] - self.ev_soc, 0) * P.EV["capacity_kwh"]
            penalty += EV_PENALTY_PER_KWH * shortfall

        reward = -(cost + penalty)
        info = {"cost": cost, "penalty": penalty, "grid_import": grid_import,
                "grid_export": grid_export, "hour": h}
        return self._obs(), reward, terminated, False, info

    # --------------------------------------------------------------- helpers
    @property
    def hour(self):
        return (START_HOUR + self.t) % 24

    def _obs(self):
        h = self.hour
        nxt = (h + 1) % 24
        weather_onehot = [float(self.day["weather"] == w) for w in P.WEATHER_TYPES]
        pending = [float(self.app_status[n] == "pending") for n in self.appliance_names]
        obs = [
            np.sin(2 * np.pi * h / 24), np.cos(2 * np.pi * h / 24),
            self.battery_soc, self.ev_soc, float(self.day["ev_plugged_in"][h]),
            self.day["solar"][h], self.day["solar"][nxt], self.day["load"][h],
            self.day["price"][h], self.day["price"][nxt],
            *weather_onehot, *pending,
        ]
        return np.clip(np.array(obs, dtype=np.float32),
                       self.observation_space.low, self.observation_space.high)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import profiles as P
from energy_env import EnergyEnv, decode_action

## 1. Pseudocode

**Q-learning (off-policy TD control) with discretised states and decaying ε**

| Line | Step |
|---|---|
| L1 | Initialise $Q(s,a) = 0$ for all $s \in \mathcal{S}$, $a \in \mathcal{A}$ |
| L2 | **Loop** for each episode $e = 1, \dots, E$: |
| L3 | &emsp; $\varepsilon \leftarrow \max\!\big(\varepsilon_{end},\ \varepsilon_{start} - (\varepsilon_{start}-\varepsilon_{end})\cdot e / (f \cdot E)\big)$ |
| L4 | &emsp; Reset environment, observe $o$; $S \leftarrow \text{discretise}(o)$ |
| L5 | &emsp; **Loop** for each step of the episode: |
| L6 | &emsp;&emsp; Select $A$ from $S$ using $\varepsilon$-greedy policy derived from $Q$ |
| L7 | &emsp;&emsp; Take action $A$, observe $R$, $o'$ |
| L8 | &emsp;&emsp; $S' \leftarrow \text{discretise}(o')$ |
| L9 | &emsp;&emsp; target $\leftarrow R$ if $S'$ is terminal, otherwise $R + \gamma \max_a Q(S', a)$ |
| L10 | &emsp;&emsp; $Q(S,A) \leftarrow Q(S,A) + \alpha\,[\text{target} - Q(S,A)]$ |
| L11 | &emsp;&emsp; $S \leftarrow S'$ |
| L12 | Return greedy target policy $\pi(s) = \arg\max_a Q(s,a)$ |

Lines L1, L2, L5–L7 and L9–L12 come directly from the Practical 5 pseudocode. L3 (ε decay), L4 and L8 (discretisation) are the additions for this project. Each line is marked in the code below with a comment such as `# [L10]`.

## 2. Discretiser

The Q-table needs a finite number of states. The discretiser keeps the features that matter most for decisions and puts each into bins:

| Feature | Observation index | Bins | Values |
|---|---|---|---|
| Hour of day | 0–1 (recovered from sin/cos) | each hour | 24 |
| Battery state of charge | 2 | 0–20 %, 20–40 %, 40–60 %, 60–80 %, 80–100 % | 5 |
| EV state of charge | 3 | < 50 %, 50–65 %, 65–80 %, ≥ 80 % (target) | 4 |
| Weather | 10–12 | sunny / cloudy / rainy | 3 |
| Appliances still pending | 13–14 | 0, 1 or 2 | 3 |

Total: 24 × 5 × 4 × 3 × 3 = **4,320 states** × 12 actions = 51,840 Q-values.

Solar, load and price (indices 5–9) are left out because they are almost entirely determined by hour and weather, which are already in the state. Only the ±10 % random noise is lost. The number of pending appliances is enough to identify which one is pending, because the washing machine always starts before the dishwasher.

The EV bins are uneven on purpose: the boundary at 80 % matches the departure target, so the agent can tell exactly when the EV penalty would be avoided.

In [ ]:
N_HOUR, N_BATT, N_EV, N_WEATHER, N_PENDING = 24, 5, 4, 3, 3
NUM_STATES = N_HOUR * N_BATT * N_EV * N_WEATHER * N_PENDING
NUM_ACTIONS = 12
EV_BIN_EDGES = [0.5, 0.65, 0.8]

def discretise(obs):
    """Map the 15-value observation to a single state index 0..NUM_STATES-1."""
    hour = int(round(np.arctan2(obs[0], obs[1]) * 24 / (2 * np.pi))) % 24
    batt = min(int(obs[2] * N_BATT), N_BATT - 1)          # 1.0 goes in the top bin
    ev = int(np.digitize(obs[3], EV_BIN_EDGES))
    weather = int(np.argmax(obs[10:13]))
    pending = int(obs[13] + obs[14])
    # combine the five parts into one index (like reading digits of a number)
    return (((hour * N_BATT + batt) * N_EV + ev) * N_WEATHER + weather) * N_PENDING + pending

print("Number of states:", NUM_STATES)

# check: the hour is recovered correctly for every step of an episode
env = EnergyEnv(weather="sunny")
obs, _ = env.reset(seed=0)
states = [discretise(obs)]
for _ in range(23):
    obs, *_ = env.step(5)
    states.append(discretise(obs))
print("State indices for the first 6 hours:", states[:6])

## 3. Q-learning agent

In [ ]:
def epsilon_greedy(Q, state, epsilon, rng):
    """[L6] Random action with probability epsilon, otherwise the best action (ties broken randomly)."""
    if rng.random() < epsilon:
        return int(rng.integers(Q.shape[1]))
    q = Q[state]
    best_actions = np.flatnonzero(q == q.max())
    return int(rng.choice(best_actions))


def Qlearning(env, gamma=1.0, alpha=0.3, eps_start=1.0, eps_end=0.05,
              eps_decay_frac=0.8, max_episodes=50000, seed=0):
    rng = np.random.default_rng(seed)

    # [L1] initialise action-value table
    Q = np.zeros((NUM_STATES, NUM_ACTIONS))
    episode_returns = np.zeros(max_episodes)

    # [L2] loop over episodes (one episode = one simulated day)
    for e in range(max_episodes):
        # [L3] linearly decaying epsilon (GLIE-style), constant after eps_decay_frac of training
        epsilon = max(eps_end, eps_start - (eps_start - eps_end) * e / (eps_decay_frac * max_episodes))

        # [L4] reset and discretise the first observation
        obs, _ = env.reset(seed=seed * 1_000_000 + e)
        state = discretise(obs)
        terminated = truncated = False
        G = 0.0

        # [L5] loop over the 24 hourly steps
        while not (terminated or truncated):
            # [L6] behaviour policy: epsilon-greedy w.r.t. Q
            action = epsilon_greedy(Q, state, epsilon, rng)

            # [L7] act, observe reward and next observation
            next_obs, reward, terminated, truncated, _ = env.step(action)

            # [L8] discretise the next observation
            next_state = discretise(next_obs)

            # [L9] TD target; Q(terminal, .) = 0 so there is no bootstrap at the end of the day
            target = reward if terminated else reward + gamma * np.max(Q[next_state])

            # [L10] Q-learning update
            Q[state, action] += alpha * (target - Q[state, action])

            # [L11] move to the next state
            state = next_state
            G += reward

        episode_returns[e] = G

    # [L12] greedy target policy
    pi = np.argmax(Q, axis=1)
    return Q, pi, episode_returns

## 4. Evaluation and baselines

Evaluation runs a policy **greedily** (no exploration) on 300 fixed test days that were never used in training (different seeds). The same test days are used for every policy, so the comparison is fair.

- **Random policy:** uniform over the 12 actions.
- **Rule-based policy:** charge the battery when solar exceeds load, discharge during the evening peak; charge the EV when plugged in outside the peak; start appliances during the cheap midday period.

In [ ]:
TEST_SEED_OFFSET = 10_000_000

def evaluate(policy, n_episodes=300, weather=None):
    """policy(obs) -> action. Returns mean and std of the episode return."""
    env = EnergyEnv(weather)
    returns = []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=TEST_SEED_OFFSET + ep)
        G, done = 0.0, False
        while not done:
            obs, r, done, _, _ = env.step(policy(obs))
            G += r
        returns.append(G)
    return np.mean(returns), np.std(returns)

_rng = np.random.default_rng(0)
def random_policy(obs):
    return int(_rng.integers(NUM_ACTIONS))

def rule_policy(obs):
    solar, load, price, plugged = obs[5], obs[7], obs[8], obs[4]
    battery = 0 if solar > load else (2 if price >= P.PRICE_PEAK else 1)
    ev = 1 if plugged and price < P.PRICE_PEAK else 0
    appliance = 1 if price <= P.PRICE_MIDDAY else 0
    return battery * 4 + ev * 2 + appliance

def greedy_policy_from(pi):
    return lambda obs: int(pi[discretise(obs)])

rule_mean, rule_std = evaluate(rule_policy)
print(f"Rule-based baseline: {rule_mean:.2f} ± {rule_std:.2f}")

## 5. Main training run
Hyperparameters chosen from the sweep in Section 8: α = 0.3 (best mean and lowest spread across seeds), γ = 1 (undiscounted, as the episode is a fixed 24 hours), ε from 1.0 to 0.05 over the first 80 % of episodes, 50,000 episodes.

In [ ]:
start = time.time()
Q, pi, train_returns = Qlearning(EnergyEnv(), alpha=0.3, gamma=1.0, max_episodes=50000, seed=0)
print(f"Training time: {time.time() - start:.0f} s")

window = 500
moving_avg = np.convolve(train_returns, np.ones(window) / window, mode="valid")
plt.figure(figsize=(9, 4))
plt.plot(moving_avg, label=f"Q-learning (moving average, {window} episodes)")
plt.axhline(rule_mean, color="r", linestyle="--", label="rule-based baseline")
plt.xlabel("episode (day)"); plt.ylabel("episode return (−daily cost, $)")
plt.title("Q-learning training curve"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

The training curve includes exploration (ε > 0), so it sits below the greedy test performance. The early part is close to the random policy; the return rises as ε decreases and the Q-values improve.

## 6. Results: Q-learning vs baselines, overall and per weather

In [ ]:
policies = {"random": random_policy, "rule-based": rule_policy, "Q-learning": greedy_policy_from(pi)}
print(f"{'policy':<12}" + "".join(f"{w:>16}" for w in ["mixed", "sunny", "cloudy", "rainy"]))
for name, pol in policies.items():
    row = f"{name:<12}"
    for w in [None] + P.WEATHER_TYPES:
        m, s = evaluate(pol, weather=w)
        row += f"{m:>9.2f} ± {s:<4.2f}"
    print(row)

## 7. What did the agent learn?
One sunny and one rainy test day, hour by hour, using the greedy policy.

In [ ]:
ACTION_NAMES = {"battery": ["charge", "idle", "discharge"], "ev": ["wait", "charge"], "app": ["defer", "start"]}

def show_day(weather, seed=TEST_SEED_OFFSET):
    env = EnergyEnv(weather)
    obs, _ = env.reset(seed=seed)
    print(f"--- {weather} day ---")
    print("hour | battery    EV      appliance | batt  EV   | price | reward")
    done, total = False, 0.0
    while not done:
        a = int(pi[discretise(obs)])
        b, ev, ap = decode_action(a)
        price = obs[8]
        obs, r, done, _, info = env.step(a)
        total += r
        print(f" {info['hour']:02d}  | {ACTION_NAMES['battery'][b]:<10} {ACTION_NAMES['ev'][ev]:<7} {ACTION_NAMES['app'][ap]:<9} "
              f"| {obs[2]:.2f}  {obs[3]:.2f} | {price:.2f}  | {r:6.2f}")
    print(f"Total return: {total:.2f}\n")

show_day("sunny")
show_day("rainy")

## 8. Multiple runs and hyperparameter tuning

Each configuration is trained with 3 random seeds and evaluated greedily on the same 300 test days. Results are reported as mean ± standard deviation **across seeds**, which shows how reliable the training is, not just how good one lucky run was.

Swept: learning rate α ∈ {0.05, 0.1, 0.2, 0.3}. This takes about 15–20 minutes in Colab. Set `RUN_SWEEP = False` to skip it.

In [ ]:
RUN_SWEEP = True
SWEEP_ALPHAS = [0.05, 0.1, 0.2, 0.3]
SWEEP_SEEDS = [0, 1, 2]
SWEEP_EPISODES = 30000

if RUN_SWEEP:
    sweep_results = {}
    for alpha in SWEEP_ALPHAS:
        scores = []
        for seed in SWEEP_SEEDS:
            _, pi_s, _ = Qlearning(EnergyEnv(), alpha=alpha, max_episodes=SWEEP_EPISODES, seed=seed)
            scores.append(evaluate(greedy_policy_from(pi_s))[0])
        sweep_results[alpha] = scores
        print(f"alpha = {alpha:<5} mean return = {np.mean(scores):7.2f} ± {np.std(scores):.2f}   (per seed: {np.round(scores, 2)})")

    plt.figure(figsize=(6, 4))
    means = [np.mean(sweep_results[a]) for a in SWEEP_ALPHAS]
    stds = [np.std(sweep_results[a]) for a in SWEEP_ALPHAS]
    plt.errorbar(range(len(SWEEP_ALPHAS)), means, yerr=stds, fmt="o-", capsize=5, label="Q-learning")
    plt.axhline(rule_mean, color="r", linestyle="--", label="rule-based baseline")
    plt.xticks(range(len(SWEEP_ALPHAS)), SWEEP_ALPHAS)
    plt.xlabel("learning rate α"); plt.ylabel("mean test return")
    plt.title(f"Learning rate sweep ({len(SWEEP_SEEDS)} seeds, {SWEEP_EPISODES} episodes)")
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 9. Notes for the report

Points to discuss, depending on the results above:

- **Discretisation trade-off.** 5 battery bins means 18 % and 2 % charge look identical to the agent. Coarse bins keep the table small enough to learn, but limit how precisely the agent can plan. This is the main motivation for DQN (Agent 2), which uses the full continuous observation.
- **Uneven experience across weather.** Rainy days occur 10 % of the time, so rainy-day states are visited about 6 times less often than sunny ones and their Q-values are less accurate. Possible fixes: train with equal weather probabilities, or more episodes.
- **Delayed EV penalty.** The EV shortfall penalty arrives only at the last step (07:00), but depends on charging decisions made hours earlier. With γ = 1 the penalty propagates back through the TD updates, but slowly, one step per visit.
- **Variance between seeds.** The spread in Section 8 shows how sensitive tabular Q-learning is to the random exploration history. A higher α learns faster but makes noisier updates in this stochastic environment.
- **Behaviour in Section 7.** Check whether the agent discharges the battery during the 3pm–9pm peak and whether the EV reaches 80 % by 07:00. In our test run it often kept a full battery through the peak and left the EV slightly short (about 75 %), paying a small penalty. Both are signs of the limits above: the 65–80 % EV bin hides how close the car is to the target, and late-evening states are learned from delayed rewards.
- **Possible improvements:** finer battery bins near empty/full, adding a price-period feature, more training episodes, a decaying α, or Double Q-learning to reduce overestimation from the max operator.